# MDBind: Custom Protein-Ligand Binding Site Screening

MDBind is a high-performance interactive tool for predicting protein-ligand binding sites. This specific Colab Notebook is engineered for **Custom Structure Uploads and High-Throughput Ligand Screening**.

Users can upload their proprietary `.pdb` structures and evaluate them against specific ligands or a built-in library of approved drugs (e.g., DrugBank). The pipeline integrates multimodal representation learning (Ankh and UniMol), evidential deep learning (EDL) for epistemic uncertainty quantification, and an interactive 3D visualization dashboard. For high-throughput screening, the algorithm performs a max-pooling operation across all ligands, mapping the highest probable interacting drug and its corresponding uncertainty to each target residue.

In [ ]:
#@title **1. Install Dependencies and Configure Environment (~3-5 mins)**
#@markdown Executing this cell will automatically configure the underlying PyTorch/CUDA environment, install required Python dependencies, and fetch external computational tools (e.g., mkdssp, msms) along with pre-trained feature normalization files.
import os
import sys

print("--> [Step 1/3] Installing core Python dependencies...")
!pip install -q biopython transformers==4.39.3 unimol_tools rdkit networkx py3Dmol plotly pandas tqdm openpyxl scipy scikit-learn periodictable

print("--> [Step 2/3] Cloning MDBind repository (fetching core modules: model.py, utils.py, readData.py)...")
GITHUB_REPO = "https://github.com/zhaoqichang/MDBind.git"
if not os.path.exists("MDBind_repo"):
    os.system(f"git clone {GITHUB_REPO} MDBind_repo")
    os.system("cp -r MDBind_repo/* .")

print("--> [Step 3/3] Granting execution permissions to topological analysis tools...")
if os.path.exists("tools/mkdssp"):
    !chmod +x tools/mkdssp
if os.path.exists("tools/msms"):
    !chmod +x tools/msms

print("\n🎉 Environment initialization and dependency configuration completed successfully!")

In [ ]:
#@title **2. Initialize Core Deep Learning Models**
#@markdown This cell loads the protein large language model (Ankh-Large) and the molecular representation model (UniMol) from pre-trained hubs, and automatically initializes the weights for the MDBind ensemble architecture.
import os
import torch
import numpy as np
import warnings
from transformers import AutoTokenizer, T5EncoderModel
from unimol_tools import UniMolRepr
from model import MDBind

warnings.filterwarnings("ignore")
DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"[Device Status] Current acceleration device in use: {DEVICE}")

# Common model configuration parameters
nn_config = {
    'dssp_exec': './tools/mkdssp' if os.path.exists('./tools/mkdssp') else 'mkdssp',
    'msms_exec': './tools/msms' if os.path.exists('./tools/msms') else 'msms',
    'rfeat_dim': 1556, 'ligand_dim': 512, 'hidden_dim': 256, 'heads': 4,
    'augment_eps': 0.0, 'rbf_num': 8, 'top_k': 30, 'attn_drop': 0.0,
    'dropout': 0.0, 'num_layers': 5, 'max_distance': 15,
}

# Check for global feature normalization matrices
for feat_file in ['./tools/dssp_max_repr.npy', './tools/dssp_min_repr.npy', './tools/ankh_max_repr.npy', './tools/ankh_min_repr.npy']:
    if not os.path.exists(feat_file):
        print(f"[Warning] Missing statistical normalization file {feat_file}.")

print("\n[1/3] Loading pre-trained protein large language model Ankh-Large...")
ankh_path = 'ElnaggarLab/ankh-large'
tokenizer = AutoTokenizer.from_pretrained(ankh_path)
ankh_model = T5EncoderModel.from_pretrained(ankh_path).to(DEVICE)
ankh_model.eval()

print("[2/3] Loading multimodal molecular representation model UniMol...")
unimol_clf = UniMolRepr(data_type='molecule', remove_hs=True, model_name='unimolv1', model_size='164')

print("[3/3] Dynamically constructing the MDBind 3-Fold ensemble network...")
def load_ensemble_models(weights_dir="./weights", num_folds=3):
    from collections import OrderedDict

    ensemble_list = []
    for fold in range(num_folds):
        model = MDBind(
            rfeat_dim=nn_config['rfeat_dim'], ligand_dim=nn_config['ligand_dim'],
            hidden_dim=nn_config['hidden_dim'], heads=nn_config['heads'],
            augment_eps=nn_config['augment_eps'], rbf_num=nn_config['rbf_num'],
            top_k=nn_config['top_k'], attn_drop=nn_config['attn_drop'],
            dropout=nn_config['dropout'], num_layers=nn_config['num_layers']
        ).to(DEVICE)

        ckpt_path = os.path.join(weights_dir, f'fold{fold}.ckpt')
        if os.path.exists(ckpt_path):
            state_dict = torch.load(ckpt_path, map_location=DEVICE)

            new_state_dict = OrderedDict()
            for k, v in state_dict.items():
                name = k[7:] if k.startswith('module.') else k
                new_state_dict[name] = v

            model.load_state_dict(new_state_dict)
            print(f"   -> Successfully loaded weights for Fold {fold}: {ckpt_path}")
        else:
            print(f"   -> {ckpt_path} not found. Proceeding with random weights for testing.")

        model.eval()
        ensemble_list.append(model)
    return ensemble_list

models = load_ensemble_models()
print("\n🌟 All core models have been successfully initialized!")

In [ ]:
#@title **3. Custom Protein Inference & Ligand Screening** 🔬
target_chain = "A" #@param {type:"string"}
ligand_smiles = "" #@param {type:"string"}
#@markdown - Run this cell and click **"Choose Files"** to upload your local `.pdb` structure.
#@markdown - If `ligand_smiles` is left blank, the system will automatically parse `./datasets/drugbank_approved_smiles.txt` and load precomputed features from `./datasets/drugs.pkl` for ultra-fast batch screening.

import os
import math
import pickle
import numpy as np
import pandas as pd
from Bio.PDB import PDBParser
from Bio.PDB.DSSP import DSSP
from scipy.spatial import distance_matrix
from google.colab import data_table, files
from readData import shortest_path_matrix_from_smiles_no_hs, pad_matrix_to_size, mapSS, calc_pseudo_cb, get_chi1_angle
from utils import calMass
from tqdm import tqdm

def normalize_feat(arr, max_path, min_path):
    if os.path.exists(max_path) and os.path.exists(min_path):
        max_v = np.load(max_path)
        min_v = np.load(min_path)
        if max_v.ndim > 0 and max_v.shape == arr.shape[-1:]:
            scalar = max_v - min_v
            scalar[scalar == 0] = 1.0
            return (arr - min_v) / scalar
    return arr

# 1. Structure Upload
print("Please upload your target PDB file (1 file only):")
uploaded = files.upload()
if not uploaded:
    raise ValueError("No file detected!")
pdb_file = list(uploaded.keys())[0]
target_id = pdb_file.split('.')[0]
target_chain = target_chain.strip().upper()

# 2. Extract Protein Topological Features
print("Parsing PDB topology and extracting DSSP/Center-of-Mass features...")
parser = PDBParser(QUIET=True)
structure = parser.get_structure("target", pdb_file)
model_obj = structure[0]
chain_obj = model_obj[target_chain]

try:
    dssp_inst = DSSP(model_obj, pdb_file, dssp=nn_config['dssp_exec'])
    dssp_keys = set(dssp_inst.keys())
except Exception:
    dssp_inst, dssp_keys = {}, set()

dssp_res, xyz_res, ca_list, cb_list, angles_list, res_ids, res_names = [], [], [], [], [], [], []
chain_atoms = ['N', 'CA', 'C', 'O']
residues = sorted([r for r in chain_obj if r.id[0] == " "], key=lambda r: r.id[1])
D3TO1 = {'CYS':'C','ASP':'D','SER':'S','GLN':'Q','LYS':'K','ILE':'I','PRO':'P','THR':'T','PHE':'F','ASN':'N','GLY':'G','HIS':'H','LEU':'L','ARG':'R','TRP':'W','ALA':'A','VAL':'V','GLU':'E','TYR':'Y','MET':'M'}

for residue in residues:
    res_ids.append(residue.id[1])
    res_names.append(residue.get_resname())
    res_key = (chain_obj.id, (' ', residue.id[1], residue.id[2]))
    if res_key in dssp_keys:
        tuple_dssp = dssp_inst[res_key]
        ss_vec = mapSS.get(tuple_dssp[2], mapSS[' '])
        other_vals = [float(x) if x != "NA" else 0.0 for x in tuple_dssp[3:]]
        dssp_res.append(ss_vec + other_vals)
    else:
        dssp_res.append(np.zeros(20, dtype=float))

    line = []
    atoms = list(residue.get_atoms())
    ca_pos = residue['CA'].get_coord() if 'CA' in residue else np.zeros(3)
    pos_s, un_s = np.zeros(3), 0.0
    for atom in atoms:
        if atom.name in chain_atoms:
            line.append(atom.get_coord())
        else:
            pos_s += calMass(atom, True)
            un_s += calMass(atom, False)
    if len(line) != 4:
        line = line + [list(ca_pos)] * (4 - len(line))
    R_pos = pos_s / un_s if un_s != 0 else ca_pos
    line.append(R_pos)
    line.append(ca_pos)
    xyz_res.append(line)
    ca_list.append(ca_pos)
    if 'CB' in residue:
        cb_coord = residue['CB'].get_coord()
    elif all(atom in residue for atom in ['N', 'CA', 'C']):
        cb_coord = calc_pseudo_cb(residue['N'].get_coord(), residue['CA'].get_coord(), residue['C'].get_coord())
    else:
        cb_coord = ca_pos
    cb_list.append(cb_coord)
    chi1 = get_chi1_angle(residue)
    angles_list.append([math.sin(chi1), math.cos(chi1)] if chi1 is not None else [0.0, 0.0])

dssp_arr = np.array(dssp_res, dtype=np.float64)
xyz_arr = np.array(xyz_res, dtype=np.float64)
angles_arr = np.array(angles_list, dtype=np.float32)
ca_coords = np.array(ca_list, dtype=np.float32)
cb_coords = np.array(cb_list, dtype=np.float32)
cmap_arr = np.stack([distance_matrix(ca_coords, ca_coords), distance_matrix(cb_coords, cb_coords)], axis=-1).astype(np.float32)
seq = "".join([D3TO1.get(res.get_resname(), 'X') for res in residues])

ids = tokenizer([list(seq)], add_special_tokens=True, padding=True, is_split_into_words=True, return_tensors="pt")
with torch.no_grad():
    embedding_repr = ankh_model(input_ids=ids['input_ids'].to(DEVICE), attention_mask=ids['attention_mask'].to(DEVICE))
    ankh_arr = embedding_repr.last_hidden_state[0, :len(seq)].cpu().numpy()

dssp_norm = normalize_feat(dssp_arr, './tools/dssp_max_repr.npy', './tools/dssp_min_repr.npy')
ankh_norm = normalize_feat(ankh_arr, './tools/ankh_max_repr.npy', './tools/ankh_min_repr.npy')
min_len = min(dssp_norm.shape[0], ankh_norm.shape[0])
feature = np.concatenate([dssp_norm[:min_len], ankh_norm[:min_len]], axis=1)

rfeat_t = torch.tensor(feature, dtype=torch.float).unsqueeze(0).to(DEVICE)
xyz_t = torch.tensor(xyz_arr[:min_len], dtype=torch.float).unsqueeze(0).to(DEVICE)
mask_t = torch.ones(1, min_len, dtype=torch.long).to(DEVICE)
angles_t = torch.tensor(angles_arr[:min_len], dtype=torch.float).unsqueeze(0).to(DEVICE)
cmap_t = torch.tensor(cmap_arr[:min_len, :min_len], dtype=torch.float).unsqueeze(0).to(DEVICE)

# 3. Build Ligand Dictionary & Load Precomputed Features
smiles_dict = {}
precomputed_features = {}

if ligand_smiles.strip():
    smiles_dict["Custom_Ligand"] = ligand_smiles.strip()
else:
    drugbank_path = "./datasets/drugbank_approved_smiles.txt"
    pkl_path = "./datasets/drugs.pkl"

    # 【核心修复】Git LFS 直连穿透下载机制
    # 如果文件不存在，或者仅仅是一个 Git LFS 指针文件 (通常不到 1KB，这里设阈值1MB)，强制拉取真实数据
    if not os.path.exists(pkl_path) or os.path.getsize(pkl_path) < 1000000:
        print("Downloading precomputed features (drugs.pkl) directly via GitHub Raw/LFS URL...")
        os.makedirs("./datasets", exist_ok=True)
        # 通过 raw 链接直接穿透获取 LFS 真实大文件
        os.system(f"wget -qO {pkl_path} https://github.com/zhaoqichang/MDBind/raw/main/datasets/drugs.pkl")

    if os.path.exists(pkl_path):
        try:
            with open(pkl_path, "rb") as f:
                precomputed_features = pickle.load(f)
            print(f"✅ Successfully loaded {len(precomputed_features)} precomputed ligand features from {pkl_path}")
        except Exception as e:
            print(f"⚠️ Warning: Failed to load {pkl_path}. Falling back to real-time UniMol computation. Error: {e}")

    if os.path.exists(drugbank_path):
        with open(drugbank_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 2 and parts[0] in precomputed_features:
                    smiles_dict[parts[0]] = parts[1]

# 4. Initialize Max-Pooling Arrays
best_probs = np.zeros(min_len)
best_uncerts = np.ones(min_len)
best_drugs = np.array(["None"] * min_len, dtype=object)

# 5. Execute Batch Screening
print(f"Executing binding site screening against {len(smiles_dict)} ligand(s)...")
for drug_id, smiles in tqdm(smiles_dict.items(), desc="Drug Screening"):
    try:
        # 智能特征路由 (Smart Feature Routing)
        if drug_id in precomputed_features:
            ligand_feat_raw = precomputed_features[drug_id]
            if isinstance(ligand_feat_raw, dict) and 'atomic_reprs' in ligand_feat_raw:
                ligand_arr = np.array(ligand_feat_raw['atomic_reprs'])
                if ligand_arr.ndim == 3 and ligand_arr.shape[0] == 1: ligand_arr = ligand_arr[0]
                ligand_feat = ligand_arr[:-1]
            else:
                ligand_feat = np.array(ligand_feat_raw)
        else:
            unimol_repr = unimol_clf.get_repr([smiles], return_atomic_reprs=True)
            atomic_reprs = unimol_repr.get('atomic_reprs', None) if isinstance(unimol_repr, dict) else unimol_repr[0].get('atomic_reprs', None)
            ligand_arr = np.array(atomic_reprs)
            if ligand_arr.ndim == 3 and ligand_arr.shape[0] == 1: ligand_arr = ligand_arr[0]
            ligand_feat = ligand_arr[:-1]

        ligand_t = torch.tensor(ligand_feat, dtype=torch.float).unsqueeze(0).to(DEVICE)

        # 拓扑图矩阵极速计算 (无需预存)
        node_paths = shortest_path_matrix_from_smiles_no_hs(smiles, max_distance=nn_config['max_distance'])
        node_paths_padded = pad_matrix_to_size(node_paths, ligand_feat.shape[0], pad_value=0)
        lig_node_paths_t = torch.tensor(node_paths_padded, dtype=torch.long).unsqueeze(0).to(DEVICE)
        lig_mask_t = torch.ones(1, ligand_feat.shape[0], dtype=torch.bool).to(DEVICE)

        # 集成推理
        evidences = []
        with torch.no_grad():
            for m in models:
                evidences.append(m(rfeat_t, xyz_t, mask_t, angles_t, cmap_t, ligand_t, lig_node_paths_t, lig_mask_t))

        avg_evidence = torch.stack(evidences, 0).mean(0)
        alpha = avg_evidence + 1.0
        S = torch.sum(alpha, dim=-1, keepdim=True)
        prob_binding = (alpha[..., 1] / S.squeeze(-1))[0].cpu().numpy()
        uncertainty = (2.0 / S.squeeze(-1))[0].cpu().numpy()

        # Max-Pooling 胜者记录
        update_mask = prob_binding > best_probs
        best_probs[update_mask] = prob_binding[update_mask]
        best_uncerts[update_mask] = uncertainty[update_mask]
        best_drugs[update_mask] = drug_id

    except Exception as e:
        pass

# 6. Render Results Table
csv_filename = f"{target_id}_{target_chain}_Screening_Results.csv"
data_output = [[target_chain, res_ids[i], res_names[i], best_probs[i], best_uncerts[i], best_drugs[i]] for i in range(min_len)]
df_results = pd.DataFrame(data_output, columns=["chain", "resi", "resn", "max_p(bind)", "uncertainty", "top_drug_id"])
df_results.to_csv(csv_filename, index=False)

data_table.enable_dataframe_formatter()
df_sorted = df_results.sort_values("max_p(bind)", ascending=False, ignore_index=True)
display(data_table.DataTable(df_sorted, min_width=150, num_rows_per_page=15))

In [ ]:
#@title **4. Interactive 3D Susceptibility Rendering (Max-Confidence)**
#@markdown This cell renders the structural "susceptibility map" resulting from the high-throughput screening. Hovering over a residue reveals the **Drug ID** that elicited the highest predicted binding probability at that specific site.
import py3Dmol
import json

def generate_colored_target_chain_pdb(in_pdb, out_pdb, chain_id, res_list, prob_list):
    prob_dict = dict(zip(res_list, prob_list))
    with open(in_pdb, 'r') as fi, open(out_pdb, 'w') as fo:
        for line in fi:
            if line.startswith("ATOM") or line.startswith("HETATM"):
                if line[21] == chain_id:
                    try:
                        r_num = int(line[22:26].strip())
                        if r_num in prob_dict:
                            b_val = (1.0 - prob_dict[r_num]) * 100.0
                            line = f"{line[:60]}{b_val:6.2f}{line[66:]}"
                    except ValueError:
                        pass
                    fo.write(line)
            elif not line.startswith("ATOM") and not line.startswith("HETATM"):
                fo.write(line)

pdb_out_filename = f"{target_id}_{target_chain}.pdb"
generate_colored_target_chain_pdb(pdb_file, pdb_out_filename, target_chain, res_ids[:min_len], best_probs)

# Construct JSON payload for visualization mapping
info_dict = {}
for i, r in enumerate(res_ids[:min_len]):
    info_dict[f"{target_chain}_{int(r)}"] = {
        "uncert": float(best_uncerts[i]),
        "drug_id": str(best_drugs[i])
    }
info_json = json.dumps(info_dict)

view = py3Dmol.view(js='https://3dmol.org/build/3Dmol.js', width=800, height=600)
with open(pdb_out_filename, "r") as f:
    view.addModel(f.read(), 'pdb')

color_scheme = {'prop': 'b', 'gradient': 'rwb', 'min': 0, 'max': 100}
view.setStyle({'chain': target_chain, 'invert': True}, {})
view.setStyle({'chain': target_chain}, {'cartoon': {'colorscheme': color_scheme}})

for idx, p_val in enumerate(best_probs):
    if p_val > 0.35:
        r_id = int(res_ids[idx])
        view.addStyle({'and': [{'chain': target_chain}, {'resi': r_id}, {'resn': ["GLY", "PRO"], 'invert': True}, {'atom': ['C', 'O', 'N'], 'invert': True}]},
                      {'stick': {'colorscheme': color_scheme, 'radius': 0.32}})

hover_func = """function(atom,viewer,event,container){
    if(!atom.label){
        var info_data = %s;
        var key = atom.chain + "_" + atom.resi;
        var node_info = info_data[key] || {uncert: 0, drug_id: "N/A"};

        var real_prob = 1.0 - (atom.b / 100.0);
        var text = atom.chain+"/"+atom.resi+"/"+atom.resn+" | MaxP=" + real_prob.toFixed(3) +
                   " | Uncert=" + node_info.uncert.toFixed(3) +
                   " | Top Drug=" + node_info.drug_id;

        atom.label=viewer.addLabel(text,{position:atom,backgroundColor:'white',backgroundOpacity:0.85,borderColor:'black',borderThickness:1.5,fontColor:'black'});
    }
}""" % (info_json)

unhover_func = """function(atom,viewer){if(atom.label){viewer.removeLabel(atom.label);delete atom.label;}}"""
view.setHoverable({}, True, hover_func, unhover_func)

view.zoomTo({'chain': target_chain})
view.show()

In [ ]:
#@title **5. Download Screening Results**
#@markdown Executing this block packages the screening CSV (containing `max_p(bind)`, `uncertainty`, and `top_drug_id`) along with the colored single-chain 3D PDB model into a Zip archive for direct download.
from google.colab import files
import os

zip_filename = f"{target_id}_{target_chain}_Screening_Predictions.zip"
csv_filename = f"{target_id}_{target_chain}_Screening_Results.csv"
pdb_out_filename = f"{target_id}_{target_chain}.pdb"

os.system(f"zip -q -r {zip_filename} {pdb_out_filename} {csv_filename}")

if os.path.exists(zip_filename):
    files.download(zip_filename)
    print(f"--> Archive successfully generated. Download triggered for [{zip_filename}]!")
else:
    print("--> [Error] Core output files not found in the current workspace.")